# Parsing the Logfiles

In [14]:
import json
from pathlib import Path
import pandas as pd

ROUND = 1
TIMESTAMP = "0416-0300"


LOG_DIR = Path(f"../logs/round_{ROUND}/{TIMESTAMP}")

# .json has round, status, profit, activitiesLog
with open(LOG_DIR / f"{TIMESTAMP}.json") as f:
    submission = json.load(f)

print(f"Round: {submission['round']}, Status: {submission['status']}, Profit: {submission['profit']}")

# Parse activitiesLog into DataFrame
from io import StringIO
activities = pd.read_csv(StringIO(submission["activitiesLog"]), sep=";")

sub_osmium = activities[activities["product"] == "ASH_COATED_OSMIUM"].copy()
sub_pepper = activities[activities["product"] == "INTARIAN_PEPPER_ROOT"].copy()

print(f"\nFinal PnL:")
for name, df in [("OSMIUM", sub_osmium), ("PEPPER", sub_pepper)]:
    last = df.groupby("day").last()["profit_and_loss"]
    print(f"  {name}: {last.values}")

Round: 1, Status: FINISHED, Profit: 10001.5625

Final PnL:
  OSMIUM: [2715.5625]
  PEPPER: [7286.]


In [15]:
with open(LOG_DIR / f"{TIMESTAMP}.log") as f:
    log_data = json.load(f)

fills = pd.DataFrame(log_data["tradeHistory"])
fills = fills[fills["buyer"].eq("SUBMISSION") | fills["seller"].eq("SUBMISSION")].copy()
fills["side"] = fills["buyer"].apply(lambda x: "BUY" if x == "SUBMISSION" else "SELL")

print(f"Total fills: {len(fills)}")
fills.groupby(["symbol", "side"]).agg(count=("quantity", "size"), total_qty=("quantity", "sum"), avg_price=("price", "mean")).round(2)

Total fills: 89


count  total_qty  avg_price
symbol               side                             
ASH_COATED_OSMIUM    BUY      41        245    9993.68
                     SELL     43        279   10004.93
INTARIAN_PEPPER_ROOT BUY       5         80   12008.40